# Twitch por dentro
## Proyecto de análisis de redes con SageMath

**Pregunta guía:** ¿cómo se reparte la popularidad, quién conecta zonas distintas de la red y qué tan agrupados están los usuarios que transmiten en portugués?

Este cuaderno es el proyecto de análisis, no la memoria escrita. Va paso a paso y explica cada concepto antes de usarlo.

**Datos incluidos en el proyecto:** la carpeta `twitch_gamers/` contiene la red completa y sus características de usuario. Primero filtramos por la columna `language == PT`; después conservamos solo las amistades cuyos dos extremos quedaron en ese grupo. Ese filtrado produce la red portuguesa analizada aquí.

**Cómo ejecutarlo:** abre este notebook con SageMath/Jupyter y ejecuta las celdas en orden. La primera sección vuelve a hacer el filtro desde los CSV originales y guarda el resultado dentro de `datos/`.

**Paleta visual:** burdeos `#9D2235` y albero `#FFC72C`.


## 0. Preparar la red portuguesa

La carpeta original contiene dos CSV: uno describe usuarios y otro relaciones. Usaremos `language` para seleccionar usuarios con idioma `PT`. Luego filtraremos las relaciones para quedarnos con las que unen **dos** usuarios PT. Así obtenemos el subgrafo inducido por los usuarios portugueses.

Conservamos también a los usuarios PT que no tengan enlaces PT; aparecerán como vértices aislados. El filtrado se guarda como dos CSV pequeños dentro de `datos/`, para poder revisar exactamente qué entró en el análisis.


In [ ]:
import csv, os
from pathlib import Path

CARPETA_ORIGINAL = Path("twitch_gamers")
ARCHIVO_USUARIOS = CARPETA_ORIGINAL / "large_twitch_features.csv"
ARCHIVO_RELACIONES = CARPETA_ORIGINAL / "large_twitch_edges.csv"
CARPETA_DATOS = Path("datos")
CARPETA_DATOS.mkdir(exist_ok=True)
CSV_PT_NODOS = CARPETA_DATOS / "twitch_PT_nodes.csv"
CSV_PT_ARISTAS = CARPETA_DATOS / "twitch_PT_edges.csv"

if not ARCHIVO_USUARIOS.exists() or not ARCHIVO_RELACIONES.exists():
    raise FileNotFoundError(
        "No encuentro twitch_gamers/large_twitch_features.csv y large_twitch_edges.csv. "
        "Coloca la carpeta twitch_gamers en la raíz del proyecto."
    )

# Paso 1: formar el conjunto de IDs cuya característica language vale PT.
usuarios_pt = set()
conteo_idiomas = {}
with ARCHIVO_USUARIOS.open(newline="", encoding="utf-8-sig") as archivo:
    for fila in csv.DictReader(archivo):
        idioma = fila["language"].strip().upper()
        conteo_idiomas[idioma] = conteo_idiomas.get(idioma, 0) + 1
        if idioma == "PT":
            usuarios_pt.add(int(fila["numeric_id"]))

# Paso 2: conservar solamente aristas PT—PT.
aristas_pt = []
with ARCHIVO_RELACIONES.open(newline="", encoding="utf-8-sig") as archivo:
    for fila in csv.DictReader(archivo):
        u = int(fila["numeric_id_1"])
        v = int(fila["numeric_id_2"])
        if u in usuarios_pt and v in usuarios_pt:
            aristas_pt.append((u, v))

# Guardamos los datos derivados para que el filtrado sea visible y reproducible.
with CSV_PT_NODOS.open("w", newline="", encoding="utf-8") as archivo:
    escritor = csv.writer(archivo)
    escritor.writerow(["numeric_id", "language"])
    escritor.writerows((u, "PT") for u in sorted(usuarios_pt))
with CSV_PT_ARISTAS.open("w", newline="", encoding="utf-8") as archivo:
    escritor = csv.writer(archivo)
    escritor.writerow(["numeric_id_1", "numeric_id_2"])
    escritor.writerows(aristas_pt)

print("Usuarios en el CSV original:", sum(conteo_idiomas.values()))
print("Idiomas disponibles:", sorted(conteo_idiomas))
print("Usuarios con language = PT:", len(usuarios_pt))
print("Aristas originales que unen dos usuarios PT:", len(aristas_pt))
print("CSV filtrados guardados en:", CARPETA_DATOS.resolve())

## 1. Convertir las amistades filtradas en un grafo

Un **grafo** representa relaciones: los usuarios son puntos (vértices) y cada amistad mutua es una línea (arista). Como la relación es mutua, la línea no lleva flecha.

Cargamos primero todos los IDs PT y luego las aristas filtradas. Esto mantiene dentro del grafo los usuarios portugueses que no tienen amistades con otros usuarios portugueses.

El **grado** de un vértice es su número de amistades. El lema del apretón de manos dice que al sumar todos los grados cada amistad se cuenta dos veces; por eso la suma debe ser `2 × número de aristas`.


In [ ]:
usuarios_pt = []
with CSV_PT_NODOS.open(newline="", encoding="utf-8") as archivo:
    usuarios_pt = [int(fila["numeric_id"]) for fila in csv.DictReader(archivo)]
aristas_pt = []
with CSV_PT_ARISTAS.open(newline="", encoding="utf-8") as archivo:
    aristas_pt = [(int(fila["numeric_id_1"]), int(fila["numeric_id_2"]))
                  for fila in csv.DictReader(archivo)]

G = Graph()
G.add_vertices(usuarios_pt)
G.add_edges(aristas_pt)
n = int(G.order())
m = int(G.size())
grados = [int(G.degree(v)) for v in G.vertices()]
suma_grados = sum(grados)
densidad = 2*m/(n*(n-1))
aislamiento = sum(1 for d in grados if d == 0)

print(f"Usuarios portugueses (vértices): {n:,}")
print(f"Amistades PT—PT (aristas): {m:,}")
print(f"Usuarios aislados dentro del grupo PT: {aislamiento}")
print(f"Grado medio: {sum(grados)/n:.2f} amistades PT por usuario")
print(f"Grado mínimo / máximo: {min(grados)} / {max(grados)}")
print(f"Densidad: {100*densidad:.3f}% de las amistades posibles entre usuarios PT")
print(f"Apretón de manos: {suma_grados:,} = 2 × {m:,} = {2*m:,} → {suma_grados == 2*m}")
print("¿Tiene lazos?:", G.has_loops())
print("¿Tiene aristas múltiples?:", G.has_multiple_edges())
print("Los datos derivan de twitch_gamers al filtrar language = PT.")

### Una primera imagen de la red filtrada

Dibujar todos los puntos a la vez puede parecer una maraña. Vemos una muestra aleatoria de 120 usuarios PT para intuir la forma de la red. Los usuarios aislados también pueden aparecer. La imagen ayuda a orientarse; las conclusiones numéricas se calculan con la red portuguesa completa.


In [ ]:
set_random_seed(2026)
muestra = sample(list(G.vertices()), min(120, n))
G_muestra = G.subgraph(muestra)
G_muestra.plot(vertex_size=18, vertex_color="#9D2235", edge_color="#C9B7B0",
               layout="spring", title="Una muestra de 120 usuarios de Twitch PT").show(figsize=8)

## 2. ¿La mayoría tiene pocas amistades PT y unos pocos muchísimas?

El grado que usamos aquí cuenta solo las amistades que quedan **después del filtro PT—PT**. Contamos cuántos usuarios hay de cada grado y ajustamos tres curvas, como en la Práctica 2: exponencial, potencial y potencial desplazada.

`find_fit` busca parámetros que reduzcan el error cuadrático. `initial_guess` da un punto de partida al algoritmo. El ajuste describe la forma de estos datos, pero por sí solo no prueba cómo se creó la red ni demuestra el “efecto Mateo”.


In [ ]:
frecuencias = {k: grados.count(k) for k in set(grados)}
datos_grado = sorted((int(k), int(f)) for k,f in frecuencias.items() if k > 0)
var('a b c x')
modelo_exp(x) = a*exp(b*x)
modelo_pot(x) = a*x^b
modelo_desplazado(x) = a*abs(x-c)^b

guess_a = max(frecuencias.values())
p_exp = find_fit(datos_grado, modelo_exp, initial_guess=[guess_a,-0.1], solution_dict=True)
p_pot = find_fit(datos_grado, modelo_pot, initial_guess=[guess_a,-1.0], solution_dict=True)
p_des = find_fit(datos_grado, modelo_desplazado, initial_guess=[guess_a,-1.0,0.0], solution_dict=True)

f_exp(x) = modelo_exp(x).subs(p_exp)
f_pot(x) = modelo_pot(x).subs(p_pot)
f_des(x) = modelo_desplazado(x).subs(p_des)
def error_cuadratico(modelo, parametros):
    return float(sum((y-float(modelo(k).subs(parametros)))^2 for k,y in datos_grado))

print("Error cuadrático exponencial:", round(error_cuadratico(modelo_exp,p_exp),2))
print("Error cuadrático potencial:", round(error_cuadratico(modelo_pot,p_pot),2))
print("Error cuadrático desplazado:", round(error_cuadratico(modelo_desplazado,p_des),2))
print("Parámetros potencial:", {str(k):round(float(v),4) for k,v in p_pot.items()})

puntos = point(datos_grado, color="#9D2235", size=18, legend_label="Datos PT")
curvas = plot(f_exp,(x,1,max(grados)),color="#555555",legend_label="Exponencial")
curvas += plot(f_pot,(x,1,max(grados)),color="#FFC72C",thickness=3,legend_label="Potencial")
curvas += plot(f_des,(x,1,max(grados)),color="#275DAD",linestyle="--",legend_label="Potencial desplazada")
(puntos+curvas).show(axes_labels=("Amistades k","Usuarios con k amistades"),
                      title="Distribución de amistades y ajustes", figsize=9)

### Cómo leer este gráfico

Cada punto dice cuántos usuarios tienen cierto número de amistades. Si la curva potencial representa razonablemente la cola, la red tiene una distribución desigual: muchos usuarios con pocas amistades y pocos con muchas. Comparad errores y forma visual. La práctica pide practicar el ajuste; una investigación estadística rigurosa de leyes potenciales requiere pasos adicionales, como elegir una cola y comprobar residuos.


## 3. ¿Qué cambiaría si la red portuguesa fuese aleatoria?

Creamos cuatro redes artificiales con el mismo número de usuarios PT. **GNM** conserva el número de aristas filtradas; **GNP** conserva el número esperado de aristas. **Barabási–Albert** y **Holme–Kim** añaden usuarios conectándolos preferentemente con vértices ya conectados; Holme–Kim también intenta cerrar triángulos.

Los dos últimos producen un número aproximado de aristas; imprimimos sus cifras reales para que la comparación sea transparente.


In [ ]:
set_random_seed(2026)
p = float(m)/(n*(n-1)/2)
aristas_por_nuevo = max(1,int(round(m/n)))
G_gnm = graphs.RandomGNM(int(n),int(m))
G_gnp = graphs.RandomGNP(int(n),float(p))
G_ba = graphs.RandomBarabasiAlbert(int(n),int(aristas_por_nuevo))
G_hk = graphs.RandomHolmeKim(int(n),int(aristas_por_nuevo),0.5)

def resumen(g):
    return {"usuarios":int(g.order()),"amistades":int(g.size()),
            "grado medio":float(g.average_degree()),"clustering":float(g.clustering_average())}

for nombre,g in [("Twitch PT",G),("Erdős–Rényi GNM",G_gnm),("Erdős–Rényi GNP",G_gnp),
                 ("Barabási–Albert",G_ba),("Holme–Kim",G_hk)]:
    print(nombre, resumen(g))

def perfil_grado(g):
    lista=[int(g.degree(v)) for v in g.vertices()]
    f={k:lista.count(k) for k in set(lista)}
    return sorted((k,f[k]) for k in f if k>0)

grafico = point(datos_grado,color="#9D2235",size=18,legend_label="Twitch PT")
for nombre,g,color in [("GNM",G_gnm,"#555555"),("GNP",G_gnp,"#7A9E7E"),
                       ("Barabási–Albert",G_ba,"#FFC72C"),("Holme–Kim",G_hk,"#275DAD")]:
    grafico += list_plot(perfil_grado(g),plotjoined=True,color=color,thickness=2,legend_label=nombre)
grafico.show(axes_labels=("Amistades k","Usuarios con k amistades"),
             title="Twitch comparado con cuatro modelos",figsize=9)

### Distancias entre usuarios PT

La distancia es el mínimo número de amistades necesarias para conectar dos usuarios. Si el grafo está desconectado, comparamos distancias dentro de la componente conexa más grande y mostramos qué proporción de usuarios incluye.


In [ ]:
def distancias_componente_mayor(g):
    componentes=g.connected_components()
    mayor=g.subgraph(max(componentes,key=len))
    return {"usuarios en componente mayor":int(mayor.order()),
            "porcentaje de toda la red":100*mayor.order()/g.order(),
            "distancia media":float(mayor.average_distance()),
            "diámetro":int(mayor.diameter())}
for nombre,g in [("Twitch PT",G),("GNM",G_gnm),("GNP",G_gnp)]:
    print(nombre,distancias_componente_mayor(g))

### Puente con la práctica de mundo pequeño

La práctica estudia Newman–Watts–Strogatz: empieza con un anillo y añade atajos aleatorios. El ejemplo de abajo ayuda a ver por qué unos pocos enlaces pueden reducir distancias. Es una ilustración didáctica y no un ajuste del modelo de Twitch.


In [ ]:
def Newman_Watts_Strogatz(n,k,probabilidad):
    if k%2!=0 or k<2 or k>=n:
        raise ValueError("k debe ser par y 2 <= k < n")
    H=graphs.CirculantGraph(n,list(range(1,k//2+1)))
    for u,v in list(H.edges(labels=False)):
        if random()<probabilidad:
            w=randint(0,n-1)
            if w not in [u,v] and not H.has_edge(u,w): H.add_edge(u,w)
            w=randint(0,n-1)
            if w not in [u,v] and not H.has_edge(v,w): H.add_edge(v,w)
    return H

set_random_seed(7)
anillo=graphs.CirculantGraph(80,[1,2])
atajos=Newman_Watts_Strogatz(80,4,0.12)
print("Sin atajos: aristas =",anillo.size(),"diámetro =",anillo.diameter())
print("Con atajos: aristas =",atajos.size(),"diámetro =",atajos.diameter())
(anillo.plot(vertex_size=20,vertex_color="#9D2235",edge_color="#C9B7B0",layout="circular") +
 atajos.plot(vertex_size=20,vertex_color="#FFC72C",edge_color="#9D2235",layout="circular")).show(figsize=10)

## 4. ¿Quién ocupa posiciones importantes en la red filtrada?

No hay una sola definición de “importancia”. Miramos tres propiedades calculadas en la red inducida por usuarios PT: **grado** (muchas amistades PT directas), **intermediación** (participa en muchos caminos mínimos) y **cercanía** (distancias cortas a otros usuarios de su componente).

Los identificadores son números anónimos. No se pueden convertir en nombres de canales con estos archivos.


In [ ]:
top_grado=sorted(((v,int(G.degree(v))) for v in G.vertices()),key=lambda q:q[1],reverse=True)[:5]
betweenness=G.centrality_betweenness(normalize=True)
top_intermediacion=sorted(betweenness.items(),key=lambda q:q[1],reverse=True)[:5]
componente=G.subgraph(max(G.connected_components(),key=len))
cercania=componente.centrality_closeness()
top_cercania=sorted(cercania.items(),key=lambda q:q[1],reverse=True)[:5]

print("TOP 5 — más amistades (ID, grado):")
for fila in top_grado: print(fila)
print("\nTOP 5 — más caminos mínimos atraviesan aquí (ID, intermediación):")
for v,x in top_intermediacion: print(v,round(float(x),6))
print("\nTOP 5 — más cerca del resto de su componente (ID, cercanía):")
for v,x in top_cercania: print(v,round(float(x),6))

ids_grado={v for v,d in top_grado}
ids_inter={v for v,x in top_intermediacion}
print("\nIDs que aparecen en ambos top 5:",sorted(ids_grado & ids_inter))

In [ ]:
lider=top_grado[0][0]
vecinos=G.neighbors(lider)[:30]
vecindad=G.subgraph([lider]+vecinos)
vecindad.plot(vertex_colors={"#9D2235":[lider],"#FFC72C":vecinos},
              vertex_size=300,edge_color="#BDA9A3",layout="spring",
              title=f"Entorno del usuario {lider} (máximo grado)").show(figsize=8)

## 5. ¿Hay grupos muy conectados?

Un triángulo es un grupo de tres usuarios donde todos están conectados entre sí. El **clustering medio** mide cuánto se cierran esas relaciones. Un **clique** es un grupo donde cada usuario está conectado con todos los demás. Un grafo bipartito se puede dividir en dos grupos sin aristas internas; un triángulo basta para demostrar que no es bipartito.

Buscar el clique máximo en toda la red puede costar mucho. Por eso calculamos el máximo dentro de los 200 usuarios PT con mayor grado y lo identificamos como clique del núcleo, no como máximo global. Las comunidades se estiman con modularidad greedy en el mismo núcleo.


In [ ]:
print("¿Se puede separar en dos grupos sin amistades internas (bipartito)?:",G.is_bipartite())
print("Triángulos en la red:",int(G.triangles_count()))
clustering_twitch=float(G.clustering_average())
clustering_gnp=float(G_gnp.clustering_average())
print(f"Clustering medio Twitch: {clustering_twitch:.4f}")
print(f"Clustering medio GNP:    {clustering_gnp:.4f}")
print("Un triángulo basta para demostrar que no es bipartito.")

ordenados=sorted(G.vertices(),key=lambda v:G.degree(v),reverse=True)
nucleo=ordenados[:min(200,n)]
G_nucleo=G.subgraph(nucleo)
clique=G_nucleo.clique_maximum()
print("Tamaño del clique máximo en el núcleo de 200:",len(clique))
print("Por tanto, el número cromático de G es al menos:",len(clique))

import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities
comunidades=list(greedy_modularity_communities(G_nucleo.networkx_graph()))
tamanos=sorted([len(c) for c in comunidades],reverse=True)
print("Comunidades estimadas en el núcleo:",len(comunidades))
print("Tamaño de las 8 mayores:",tamanos[:8])

if len(clique)>0:
    G_clique=G.subgraph(clique)
    G_clique.plot(vertex_color="#FFC72C",edge_color="#9D2235",vertex_size=450,
                  layout="circular",title=f"Clique máximo del núcleo: K_{len(clique)}").show(figsize=7)

### Cómo entender el resultado

Un clustering mayor que en GNP indicaría más triángulos que en una red aleatoria con densidad parecida. Eso sugiere cierre local de amistades, pero no dice qué afición une a cada grupo. En este trabajo el filtro PT procede de la columna `language`, no de la detección de juegos.


## 6. Comprobación matemática manual

Elegimos seis usuarios del núcleo PT. La matriz de adyacencia marca con 1 una amistad y con 0 su ausencia. Debe ser cuadrada, simétrica (las aristas no tienen dirección) y tener diagonal cero (no hay amistades consigo mismo). Havel–Hakimi comprueba que la secuencia de grados del subgrafo puede corresponder a un grafo.


In [ ]:
seis=nucleo[:6]
H6=G.subgraph(seis)
M=H6.adjacency_matrix()
secuencia=sorted([int(H6.degree(v)) for v in H6.vertices()],reverse=True)
print("Seis IDs:",seis)
print("Matriz de adyacencia:")
show(M)
print("¿Matriz cuadrada?:",M.nrows()==M.ncols())
print("¿Matriz simétrica?:",M==M.transpose())
print("¿Diagonal cero?:",all(M[i,i]==0 for i in range(M.nrows())))
print("Grados dentro del grupo de seis:",secuencia)

def havel_hakimi(pendientes):
    pendientes=sorted([int(d) for d in pendientes],reverse=True)
    pasos=[list(pendientes)]
    while pendientes and pendientes[0]>0:
        k=pendientes.pop(0)
        if k>len(pendientes): return pasos,False
        for i in range(k):
            pendientes[i]-=1
            if pendientes[i]<0: return pasos+[list(pendientes)],False
        pendientes=sorted(pendientes,reverse=True)
        pasos.append(list(pendientes))
    return pasos,True

pasos,es_grafica=havel_hakimi(secuencia)
print("\nPasos de Havel–Hakimi:")
for numero,paso in enumerate(pasos): print(numero,":",paso)
print("¿La secuencia es gráfica?:",es_grafica)

## 7. Resumen de esta ejecución

Las cifras se calculan con el subgrafo obtenido filtrando `language == PT` y manteniendo las aristas PT—PT.


In [ ]:
tabla_resultados=[
    ["Usuarios con language = PT", n],
    ["Aristas entre usuarios PT", m],
    ["Usuarios PT sin vecinos PT", aislamiento],
    ["Grado medio PT", round(sum(grados)/n,3)],
    ["Densidad PT (%)", round(100*densidad,4)],
    ["Grado máximo PT", max(grados)],
    ["Triángulos PT", int(G.triangles_count())],
    ["Clustering Twitch PT", round(clustering_twitch,4)],
    ["Clustering GNP", round(clustering_gnp,4)],
    ["Bipartito", G.is_bipartite()],
    ["Clique máx. en núcleo PT top 200", len(clique)],
    ["Comunidades estimadas en núcleo", len(comunidades)]
]
print("RESULTADOS DE ESTA EJECUCIÓN")
for etiqueta,valor in tabla_resultados: print(f"{etiqueta:38s} {valor}")

## Datos y referencias

- `twitch_gamers/large_twitch_features.csv`: tabla de atributos de usuario, incluida la columna `language`.
- `twitch_gamers/large_twitch_edges.csv`: relaciones de amistad mutua entre IDs.
- Los archivos filtrados `datos/twitch_PT_nodes.csv` y `datos/twitch_PT_edges.csv` se generan en la primera sección.
- B. Rozemberczki y R. Sarkar. *Twitch Gamers: a Dataset for Evaluating Proximity Preserving and Structural Role-based Node Embeddings* (2021): https://arxiv.org/abs/2101.03091
- SageMath, manual de teoría de grafos: https://doc.sagemath.org/html/en/reference/graphs/
